In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

#import numpy as np # linear algebra
#import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
def setup_environment():
    """Install Biopython and Biotite for offline submission."""
    print("Installing dependencies...")
    os.system("pip install --no-index --no-deps /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl")
    os.system("pip install --no-index --no-deps /kaggle/input/datasets/amirrezaaleyasin/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl")

setup_environment()

# =============================================================================
# 2. PHYSICAL REFINER
# =============================================================================
def adaptive_rna_constraints(coords, confidence=0.8, passes=2):
    """Refines coordinates based on RNA physical constraints (C1'-C1' distance)."""
    X = coords.copy()
    strength = max(0.6 * (1.0 - min(confidence, 0.95)), 0.05)
    
    for _ in range(passes):
        # Constraint 1: Bond distance (i to i+1) ~ 5.95A
        delta = X[1:] - X[:-1]
        dist = np.linalg.norm(delta, axis=1, keepdims=True) + 1e-6
        adjustment = delta * ((5.95 - dist) / dist) * (0.22 * strength)
        X[:-1] -= adjustment
        X[1:] += adjustment
    return X

# =============================================================================
# 3. COORDINATE CONVERSION (Integrated with Refiner)
# =============================================================================
def coords_to_rows(target_id: str, seq: str, coords: np.ndarray) -> list:
    """Converts (N_SAMPLE, L, 3) coords to submission rows with refinement."""
    rows = []
    N_SAMPLE = coords.shape[0]
    seq_len = len(seq)

    # Apply Refiner to all 5 samples
    refined_coords = np.zeros_like(coords)
    for s in range(N_SAMPLE):
        refined_coords[s] = adaptive_rna_constraints(coords[s], confidence=0.8)

    for i in range(seq_len):
        row = {"ID": f"{target_id}_{i + 1}", "resname": seq[i], "resid": i + 1}
        for s in range(N_SAMPLE):
            x, y, z = refined_coords[s, i] if s < N_SAMPLE else (0.0, 0.0, 0.0)
            row[f"x_{s + 1}"] = float(x)
            row[f"y_{s + 1}"] = float(y)
            row[f"z_{s + 1}"] = float(z)
        rows.append(row)
    return rows

# =============================================================================
# 4. MAIN INFERENCE LOOP & FILE SAVING
# =============================================================================
# Load the test data
DATA_BASE = Path("/kaggle/input/competitions/stanford-rna-3d-folding-2")
test_df = pd.read_csv(DATA_BASE / "test_sequences.csv")

all_submission_rows = []

print(f"Starting inference for {len(test_df)} sequences...")

for idx, row in test_df.iterrows():
    tid = row['target_id']
    seq = row['sequence']
    
    # --- IMPORTANT: INSERT YOUR MODEL PREDICTION HERE ---
    # Example: pred_coords = my_protenix_model.predict(seq)
    # The 'pred_coords' must be a numpy array of shape (5, len(seq), 3)
    
    # Using your actual model coordinates (currently using zeros as placeholder)
    # REPLACE THE LINE BELOW WITH YOUR MODEL OUTPUT
    pred_coords = np.zeros((5, len(seq), 3)) 
    
    # Process and refine
    target_rows = coords_to_rows(tid, seq, pred_coords)
    all_submission_rows.extend(target_rows)

# Create DataFrame and Save (Indentation is critical here!)
submission_df = pd.DataFrame(all_submission_rows)
submission_df.to_csv("submission.csv", index=False)

print("Check complete. 'submission.csv' has been written to the current directory.")

Installing dependencies...
Processing /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/amirrezaaleyasin/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
Starting inference for 28 sequences...
Check complete. 'submission.csv' has been written to the current directory.
